# Typing and modern python
* python is a "dynamically typed" language - we never HAD to write down what type a variable is, the interpreter figures it out at runtime
* but modern python (especially since 3.9/3.10) gives us optional "type hints" - little annotations that document intended types
* super important to understand right from the start: type hints are just documentation + tooling info, python itself does NOT enforce them at runtime
* in this lesson we cover type hints, modern generics, Enum, NamedTuple/dataclass, TypedDict, Protocol, match/case, the walrus operator, advanced f-strings and positional/keyword-only params

## type hints dont change runtime behaviour
* lets prove this right away with an example, because its the single most important thing to understand about python typing
* we write a function that is annotated to only accept ints, and then we call it with strings anyway

In [1]:
# we annotate "a" and "b" as int, and the return type as int too - this is just a hint, not a rule
def add(a: int, b: int) -> int:
    return a + b

# calling it "correctly" with ints works exactly as expected
print(add(40, 2))

# but python NEVER checks the annotations at runtime - the interpreter just ignores them completely
# so calling it with strings still "works", python just does whatever + does for strings (concatenation)
print(add("hello", "world"))  # --> "helloworld", no TypeError, no warning, nothing!

# this is very different from a language like Java or TypeScript where this would refuse to even compile
# in python, type hints are checked by EXTERNAL tools (mypy, pyright, ...) BEFORE you run the code, not by the interpreter itself
# those tools would flag add("hello", "world") as an error, but python itself couldnt care less
# we can even peek at the annotations python stored for us, just as plain metadata:
print(add.__annotations__)

42
helloworld
{'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}


## basic annotations
* we can annotate variables, function parameters and function return types
* the syntax is always `name: type`
* for a value that "may be None" we use `Optional[X]` (older style, needs `from typing import Optional`) or the modern `X | None` (python 3.10+, no import needed)

In [2]:
# variable annotation - just documents the intended type, we can still reassign to something else (again: not enforced!)
age: int = 30
print(age)

# a function with fully annotated parameters and return type
def greet(name: str, times: int = 1) -> str:
    return (name + " ") * times

print(greet("Alex", 3))

# Optional[X] means "X or None" - very common for values that might be missing
from typing import Optional

def find_user(user_id: int) -> Optional[str]:
    users = {1: "Alex", 2: "Mia"}
    return users.get(user_id)  # .get() returns None if the key doesnt exist, no KeyError

print(find_user(1))
print(find_user(999))

# the modern way (python 3.10+) to write the exact same thing is "X | None" - no import needed, and most people prefer this now
def find_user_modern(user_id: int) -> str | None:
    users = {1: "Alex", 2: "Mia"}
    return users.get(user_id)

print(find_user_modern(2))
print(find_user_modern(42))

30
Alex Alex Alex 
Alex
None
Mia
None


## modern generic collection syntax
* since python 3.9 we can use the builtin collection types directly as generics: `list[int]`, `dict[str, int]`, `tuple[int, str]`
* before 3.9 we had to import the capitalized versions from the `typing` module: `typing.List[int]`, `typing.Dict[str, int]`, `typing.Tuple[int, str]`
* the old `typing.List` etc still exist and still work (for backwards compatibility), but the builtin lowercase versions are preferred these days - one less import needed

In [3]:
# modern style (3.9+) - just use the builtin type directly with square brackets, no import needed at all
def average(numbers: list[int]) -> float:
    return sum(numbers) / len(numbers)

print(average([1, 2, 3, 4, 5]))

# a dict with typed keys and values
def word_lengths(words: list[str]) -> dict[str, int]:
    return {word: len(word) for word in words}

print(word_lengths(["hi", "python", "typing"]))

# a tuple with fixed, differently typed slots (a "record"-like tuple)
def make_pair(name: str, age: int) -> tuple[str, int]:
    return (name, age)

print(make_pair("Alex", 30))

# the OLD way, still works but is considered outdated style now - shown here just so you recognize it in older codebases
from typing import List, Dict, Tuple  # capitalized versions from the typing module

def average_old_style(numbers: List[int]) -> float:  # exact same meaning as list[int]
    return sum(numbers) / len(numbers)

print(average_old_style([10, 20, 30]))

3.0
{'hi': 2, 'python': 6, 'typing': 6}
('Alex', 30)
20.0


## Union types: one type OR another
* sometimes a parameter can legitimately be more than one type, for example either an int or a str
* the modern syntax (python 3.10+) is the `|` operator, e.g. `int | str`
* the older way is `typing.Union[int, str]` - still works, functionally identical, but more to type and to import

In [4]:
# modern union syntax - "value can be an int OR a str"
def parse(value: int | str) -> str:
    # we handle both possible types inside the function body ourselves - the hint doesnt do that for us!
    if isinstance(value, int):
        return f"got a number: {value}"
    return f"got text: {value}"

print(parse(42))
print(parse("hello"))

# the older equivalent using typing.Union - you will see this a lot in slightly older code
from typing import Union

def parse_old_style(value: Union[int, str]) -> str:
    if isinstance(value, int):
        return f"got a number: {value}"
    return f"got text: {value}"

print(parse_old_style(7))

got a number: 42
got text: hello
got a number: 7


## Enum - named constants
* an `Enum` (from the `enum` module) groups a fixed set of named, unique constant values under one type
* way better than using plain magic strings/numbers scattered around your code ("RED", "red", 1, ... - which one did you mean again?)
* members have a `.name` and a `.value`, we can iterate over all of them, and compare members with `is` or `==`

In [5]:
from enum import Enum

# we define our fixed set of possible states as an Enum, each member gets a name and an underlying value
class TrafficLight(Enum):
    RED = 1
    YELLOW = 2
    GREEN = 3

# accessing a member - looks almost like accessing a class attribute
current = TrafficLight.RED
print(current)          # TrafficLight.RED
print(current.name)     # "RED" - the name we chose
print(current.value)    # 1 - the underlying value we chose

# iterating over ALL members of the enum, super handy for validation or building a UI dropdown etc
for light in TrafficLight:
    print(light.name, "-->", light.value)

# comparing enum members - both "is" and "==" work here because enum members are singletons (only one instance of each ever exists)
print(current is TrafficLight.RED)   # True - "is" checks identity, works great for enums
print(current == TrafficLight.RED)   # True - "==" also works
print(current == TrafficLight.GREEN) # False

# expert note: prefer "is" for enum comparisons, its the idiomatic way and slightly faster (no need to call __eq__)

TrafficLight.RED
RED
1
RED --> 1
YELLOW --> 2
GREEN --> 3
True
True
False


## NamedTuple vs plain tuple vs dataclass
* we often need a tiny "data holder" object, for example a 2D point with x and y
* python gives us (at least) three ways to build one, each with different tradeoffs - lets build the exact same "Point" all three ways

In [6]:
# 1) plain tuple - works, but has no field names, you have to remember that index 0 is x and index 1 is y
point_tuple = (3, 4)
print(point_tuple[0], point_tuple[1])  # x, y - not very readable, easy to mix up the order by accident

# 2) typing.NamedTuple - a tuple with named fields, still immutable (cant change x/y after creation) and still indexable
from typing import NamedTuple

class PointNamedTuple(NamedTuple):
    x: int
    y: int

point_nt = PointNamedTuple(3, 4)
print(point_nt.x, point_nt.y)   # readable access by name
print(point_nt[0], point_nt[1]) # still works by index too, its a real tuple under the hood
try:
    point_nt.x = 99  # this fails! NamedTuple is immutable, just like a regular tuple
except AttributeError as error:
    print("cant modify a NamedTuple:", error)

# 3) @dataclass - a normal class, but the decorator auto-generates __init__, __repr__, __eq__ for us based on the annotations
from dataclasses import dataclass

@dataclass
class PointDataclass:
    x: int
    y: int

point_dc = PointDataclass(3, 4)
print(point_dc.x, point_dc.y)
point_dc.x = 99  # this WORKS - dataclasses are mutable by default (you can opt into frozen=True for immutability)
print(point_dc)  # auto-generated __repr__ gives us a nice readable print for free, unlike a plain class

# tradeoff summary:
# - plain tuple: lightest weight, but no names, easy to misread
# - NamedTuple: lightweight + immutable + indexable + named fields, great for simple fixed data that shouldnt change
# - dataclass: mutable by default (can be frozen), more features (default values, methods, inheritance), the modern go-to for most "data holder" classes

3 4
3 4
3 4
cant modify a NamedTuple: can't set attribute
3 4
PointDataclass(x=99, y=4)


## TypedDict - a dict with a documented shape
* `TypedDict` (from `typing`) lets us describe the expected shape of a dict - which keys exist and what type each value should have
* very useful for JSON-like data (think: an API response) so your editor and static type checkers know what to expect
* important: at runtime a TypedDict is STILL just a plain dict - python does not enforce the shape at all, its purely for documentation/tooling

In [7]:
from typing import TypedDict

# we describe the expected shape of a "Movie" dict - three keys, each with an expected type
class Movie(TypedDict):
    title: str
    year: int
    rating: float

# this matches the described shape, great
good_movie: Movie = {"title": "The Matrix", "year": 1999, "rating": 8.7}
print(good_movie)

# but nothing stops us from building a "Movie" that violates the shape completely - python wont complain at all!
bad_movie: Movie = {"title": 42, "year": "not a number", "extra_key": "not even declared"}  # a static checker would scream here, python doesnt care
print(bad_movie)

# proof that its really just a plain dict under the hood, nothing special at runtime
print(type(good_movie))       # <class 'dict'>
print(type(good_movie) is dict)  # True

{'title': 'The Matrix', 'year': 1999, 'rating': 8.7}
{'title': 42, 'year': 'not a number', 'extra_key': 'not even declared'}
<class 'dict'>
True


## Protocol - structural typing ("duck typing made explicit")
* a `Protocol` (from `typing`) describes a shape of METHODS a class needs to have, without needing to explicitly inherit from anything
* this is called "structural typing" - if it has the right methods with the right signature, it satisfies the protocol, no inheritance required
* contrast this with `abc.ABC` (abstract base classes) - there, a class MUST explicitly inherit from the abstract class to count as implementing it
* "duck typing" is the classic python saying: "if it walks like a duck and quacks like a duck, its a duck" - Protocol makes this idea explicit and checkable by tools

In [8]:
from typing import Protocol, runtime_checkable

# we define a protocol: "anything with a make_sound() method that returns a str counts as a NoiseMaker"
@runtime_checkable  # this decorator lets us actually use isinstance() checks against the protocol at runtime
class NoiseMaker(Protocol):
    def make_sound(self) -> str:
        ...  # protocols only describe the shape, they never have a real implementation

# NOTE: neither of these classes inherits from NoiseMaker in any way!
class Dog:
    def make_sound(self) -> str:
        return "woof"

class Duck:
    def make_sound(self) -> str:
        return "quack"

class Rock:
    pass  # no make_sound() method at all

def announce(thing: NoiseMaker) -> None:
    print(thing.make_sound())

# both Dog and Duck satisfy the protocol purely by HAVING the right method - no "class Dog(NoiseMaker)" needed anywhere
announce(Dog())
announce(Duck())

# isinstance() even works against the protocol at runtime (because we used @runtime_checkable), purely by checking the shape
print(isinstance(Dog(), NoiseMaker))   # True - has make_sound()
print(isinstance(Rock(), NoiseMaker))  # False - no make_sound() method at all

# compare this to abc.ABC (which we will meet properly in the object-orientation lesson): there, Dog would HAVE to
# explicitly write "class Dog(SomeAbstractBaseClass):" to count - Protocol never needs that explicit relationship,
# it only cares about the actual shape (methods) a class has, which is exactly what "structural typing" means

woof
quack
True
False


## structural pattern matching: match/case
* python 3.10 introduced `match`/`case`, at first glance it looks like a switch/case from other languages, but its way more powerful
* it doesnt just compare a single value - it can also DESTRUCTURE data (pull values out of tuples, objects, etc while matching), thats the "structural" part
* every `match` block can end with a wildcard `case _:` which matches anything (like a "default" case)

In [9]:
# 1) simple literal matching - this part really does look like a classic switch/case
def handle_command(command: str) -> str:
    match command:
        case "start":
            return "starting up..."
        case "stop":
            return "shutting down..."
        case "pause" | "hold":  # the "|" inside a case lets us match multiple literals with one case (an "or" pattern)
            return "pausing..."
        case _:  # wildcard - matches anything not matched above, similar to "default" in a switch statement
            return f"unknown command: {command}"

print(handle_command("start"))
print(handle_command("hold"))
print(handle_command("explode"))

# 2) matching on STRUCTURE - here it stops being a simple switch/case and starts destructuring data
def describe_point(point: tuple) -> str:
    match point:
        case (0, 0):                 # matches the exact shape (0, 0)
            return "origin"
        case (x, 0):                 # matches any 2-tuple where the second value is 0, and BINDS the first value to "x"
            return f"on the x-axis at {x}"
        case (0, y):                 # same idea, y-axis
            return f"on the y-axis at {y}"
        case (x, y):                 # matches any other 2-tuple, binding both values
            return f"somewhere at ({x}, {y})"
        case _:
            return "not a 2d point at all"

print(describe_point((0, 0)))
print(describe_point((5, 0)))
print(describe_point((3, 4)))

# 3) matching on a CLASS pattern, including its attributes - this is real destructuring of objects, not just values
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

def describe_point_object(point: Point) -> str:
    match point:
        case Point(x=0, y=0):               # matches a Point instance whose x AND y are both 0
            return "origin"
        case Point(x=x, y=y) if x == y:     # a class pattern can even have an extra "if" guard condition
            return f"on the diagonal at {x}"
        case Point(x=x, y=y):               # binds both attributes for any other Point
            return f"point at ({x}, {y})"
        case _:
            return "not a point at all"

print(describe_point_object(Point(0, 0)))
print(describe_point_object(Point(5, 5)))
print(describe_point_object(Point(3, 4)))

# important takeaway: a classic C/Java/C#/JS switch only ever compares one value against a list of options
# python's match/case additionally destructures shapes/objects and can bind new variables while doing so - much more expressive

starting up...
pausing...
unknown command: explode
origin
on the x-axis at 5
somewhere at (3, 4)
origin
on the diagonal at 5
point at (3, 4)


## the walrus operator :=
* the walrus operator lets us assign a value to a variable INSIDE an expression, and use that value right away
* very handy whenever we would otherwise have to compute something twice (once to check it, once to use it)
* common spots: inside a `while` loop condition, or inside a comprehension's filter condition

In [10]:
# walrus inside a while loop condition - we assign AND check in one go
data = [5, 3, 8, 2, 0, 9, 1]  # 0 acts as a "sentinel" value that tells us to stop

while (current := data.pop(0)) != 0:   # assigns current, then immediately checks "!= 0" using that same value
    print("processing:", current)
print("hit the sentinel value 0, stopping")

# without the walrus we would need something clunkier like:
#   current = data.pop(0)
#   while current != 0:
#       print("processing:", current)
#       current = data.pop(0)   # <-- have to repeat the assignment here too, easy to forget/get wrong

# walrus inside a list comprehension filter - reusing an expensive computed value without computing it twice
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# we compute "half" once per number, use it in the filter condition AND reuse the same value in the result expression
big_halves = [half for n in numbers if (half := n / 2) > 3]
print(big_halves)

# without walrus we would have to either compute n/2 twice, or write it as two separate lines with a classic loop

processing: 5
processing: 3
processing: 8
processing: 2
hit the sentinel value 0, stopping
[3.5, 4.0, 4.5, 5.0]


## advanced f-strings
* we already know basic f-strings like `f"hello {name}"`, but f-strings support a lot more formatting power
* format specs after a `:` control things like decimal places and alignment/padding
* the debug specifier `f"{value=}"` (python 3.8+) prints BOTH the expression text and its value - super handy while debugging with print statements

In [11]:
pi = 3.14159265

# ".2f" means "fixed point notation, 2 digits after the decimal point"
print(f"{pi:.2f}")

number = 42

# ">10" means "right-align inside a field of 10 characters wide" (using spaces to pad)
print(f"[{number:>10}]")

# "<10" means "left-align inside a field of 10 characters wide"
print(f"[{number:<10}]")

# we can combine width with other things, e.g. "^10" centers the value
print(f"[{number:^10}]")

# the debug specifier "=" - just add it after the expression, no format needed
# it prints "expression_text=value", great for quick debugging without writing print("pi:", pi) by hand
print(f"{pi=}")
print(f"{number=}")
print(f"{pi=:.2f}")  # the "=" can be combined with a format spec too, just put the "=" BEFORE the ":" - prints "pi=3.14" style output

3.14
[        42]
[42        ]
[    42    ]
pi=3.14159265
number=42
pi=3.14


## positional-only and keyword-only parameters
* a `/` in a function signature marks every parameter BEFORE it as positional-only - callers cant use their names, only their position
* a `*` marks every parameter AFTER it as keyword-only - callers MUST use their names, position doesnt work
* unlike type hints, this IS enforced by the interpreter at runtime - a real TypeError if you get it wrong!

In [12]:
# x and y are positional-only (before the "/"), unit is keyword-only (after the "*")
def format_point(x, y, /, *, unit="m"):
    return f"({x}{unit}, {y}{unit})"

print(format_point(3, 4))                # fine - x, y given positionally, unit uses its default
print(format_point(3, 4, unit="cm"))      # fine - unit given by keyword, exactly as required

# calling x/y by keyword fails - they are positional-ONLY, python enforces this for real (not just a hint!)
try:
    format_point(x=3, y=4)
except TypeError as error:
    print("error:", error)

# calling unit positionally also fails - it is keyword-ONLY
try:
    format_point(3, 4, "cm")
except TypeError as error:
    print("error:", error)

# why use this at all? positional-only lets you rename params later without breaking callers,
# keyword-only forces callers to be explicit about less obvious params (like "unit" here) - both improve API clarity

(3m, 4m)
(3cm, 4cm)
error: format_point() got some positional-only arguments passed as keyword arguments: 'x, y'
error: format_point() takes 2 positional arguments but 3 were given


## quick comparison to other languages
* python's type hints are optional and NOT enforced at runtime by the interpreter itself - only external tools (mypy, pyright) check them, and only before you run the code
* TypeScript hints ARE enforced, but only at compile time - the compiler strips all types away when producing plain JavaScript, so at runtime TypeScript is just as unenforced as python (the check just happens earlier, before shipping)
* Java, C#, C++, Rust and Go are on a whole different level: their types are enforced by the compiler AND still matter at runtime (they decide memory layout, method dispatch, etc) - there is no "ignore the type and see what happens" option like python's `add("hello", "world")` example at the top of this lesson
* python's `match`/`case` is structurally much closer to Rust's `match` expression (which also destructures enums/structs and binds variables while matching) than to a classic C/Java/C#/JavaScript `switch` (which can only compare a single value against constants, no destructuring)
* python's `Protocol` (structural typing) is the exact same idea as TypeScript's structural interfaces or Go's implicit interfaces - "if the shape fits, it counts", no explicit relationship needed
* Java and C# interfaces work the opposite way (structurally): a class MUST explicitly write `implements SomeInterface` / `: ISomeInterface` to count, purely having the right methods is not enough - much closer to python's `abc.ABC` than to `Protocol`

## Exercises
1. write and correctly call a function `area(width: float, height: float) -> float` that returns the rectangle area
2. define an `Enum` called `Weekday` with members for all 7 days, then write a function that uses `match`/`case` on a `Weekday` member and returns `"weekend"` for Saturday/Sunday and `"weekday"` for every other day
3. we have a plain dict-based "Book" record `{"title": "Dune", "author": "Frank Herbert", "pages": 412}` - rewrite it as a `@dataclass` called `Book` with proper type hints for all three fields, then create one instance
4. use the walrus operator inside a `while` loop that keeps drawing `random.randint(1, 10)` (print every draw) until it finally draws a value greater than 8

In [13]:
# TODO: solve exercise 1 here (area function)


# TODO: solve exercise 2 here (Weekday enum + match/case weekend/weekday)


# TODO: solve exercise 3 here (Book dataclass)


# TODO: solve exercise 4 here (walrus + random.randint while loop)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [14]:
# sample solution 1 - area function, fully annotated and called correctly
def area(width: float, height: float) -> float:
    return width * height

print(area(3.0, 4.0))

# sample solution 2 - Weekday enum + match/case
from enum import Enum

class Weekday(Enum):
    MONDAY = 1
    TUESDAY = 2
    WEDNESDAY = 3
    THURSDAY = 4
    FRIDAY = 5
    SATURDAY = 6
    SUNDAY = 7

def day_type(day: Weekday) -> str:
    match day:
        case Weekday.SATURDAY | Weekday.SUNDAY:  # "or" pattern - matches either one
            return "weekend"
        case _:
            return "weekday"

for day in Weekday:
    print(day.name, "-->", day_type(day))

# sample solution 3 - Book dataclass replacing a plain dict record
from dataclasses import dataclass

@dataclass
class Book:
    title: str
    author: str
    pages: int

book_old_style = {"title": "Dune", "author": "Frank Herbert", "pages": 412}  # the old plain-dict version, for comparison
book = Book(title="Dune", author="Frank Herbert", pages=412)  # the new dataclass version - same data, but typed and with a nice repr
print(book_old_style)
print(book)

# sample solution 4 - walrus operator inside a while loop with random.randint
import random

while (draw := random.randint(1, 10)) <= 8:
    print("drew:", draw, "--> not high enough, trying again")
print("finally drew:", draw, "--> thats greater than 8, stopping!")

12.0
MONDAY --> weekday
TUESDAY --> weekday
WEDNESDAY --> weekday
THURSDAY --> weekday
FRIDAY --> weekday
SATURDAY --> weekend
SUNDAY --> weekend
{'title': 'Dune', 'author': 'Frank Herbert', 'pages': 412}
Book(title='Dune', author='Frank Herbert', pages=412)
drew: 4 --> not high enough, trying again
finally drew: 9 --> thats greater than 8, stopping!


* congratulation you finished this lesson

## what we learned
* type hints (variables, parameters, return types) are pure documentation - python never enforces them at runtime, only external tools like mypy/pyright do
* modern generic collection syntax (`list[int]`, `dict[str, int]`, `tuple[int, str]`) replacing the older `typing.List`/`Dict`/`Tuple`
* `Optional[X]`/`X | None` and `Union[X, Y]`/`X | Y` for "maybe missing" and "one type or another"
* `Enum` for fixed sets of named constants, with `.name`, `.value`, iteration and `is`/`==` comparison
* three ways to build a small data holder: plain tuple, `NamedTuple` (immutable+indexable+named), `@dataclass` (mutable by default, more features)
* `TypedDict` documents a dict's expected shape, but its still a plain unenforced dict at runtime
* `Protocol` gives us structural typing ("duck typing made explicit") - matching methods are enough, no inheritance required, unlike `abc.ABC`
* `match`/`case` structural pattern matching - literal matching, "or" patterns, tuple/class destructuring, guard conditions and the `case _:` wildcard
* the walrus operator `:=` for assigning and using a value in the same expression (while loops, comprehensions)
* advanced f-string format specs (`.2f`, alignment) and the `{value=}` debug specifier
* positional-only (`/`) and keyword-only (`*`) parameters - one of the few typing-adjacent features that IS actually enforced by the interpreter